## 00 — Feature Properties

So far we have displayed GeoJSON on a map. Every feature renders with the same default style — blue circles for points, blue lines, blue polygons. That is not very useful.

Before we can style or filter features, we need to understand what information they carry. Every GeoJSON feature has a **`properties`** object — an arbitrary key/value store attached to that feature. This lesson is about reading it.

## Anatomy of a Feature

Every GeoJSON feature has exactly three keys:

```
feature
├── type        → always the string "Feature"
├── geometry    → where it is  (coordinates + geometry type)
└── properties  → what it is   (any key/value pairs you choose)
```

`geometry` answers the question *"where?"*  
`properties` answers the question *"what?"*

The `properties` object can hold anything: a name, a category, a population count, a color hint. It is just a dict.

In [ ]:
import json

with open("../02-Viewing_GeoJSON/data/wichita_falls.geojson") as f:
    geojson = json.load(f)

# Look at the first feature
feature = geojson["features"][0]
feature

You can see the three top-level keys — `type`, `geometry`, and `properties` — right in the output.

## Inspecting Properties

Access the `properties` dict directly. When working with an unfamiliar dataset, the first thing to do is look at the keys.

In [ ]:
# What keys does this dataset have?
feature["properties"].keys()

In [ ]:
# Print all properties for the first feature
feature["properties"]

Our dataset has two properties on every feature: `name` (a descriptive label) and `type` (a category like `"park"` or `"education"`). Real-world datasets often have dozens of properties — the workflow is the same.

## Accessing Individual Attributes

You can read a property with bracket notation or with `.get()`.

In [ ]:
# Bracket notation — raises KeyError if the key is missing
print(feature["properties"]["name"])

# .get() — returns None (or a default) if the key is missing
print(feature["properties"].get("name"))
print(feature["properties"].get("population"))        # key doesn't exist → None
print(feature["properties"].get("population", 0))     # key doesn't exist → 0

**Use `.get()` when you are not sure a property will exist on every feature.** GeoJSON properties are not required to be consistent across features — one feature might have a `population` key while another does not. Bracket notation on a missing key crashes; `.get()` returns `None` and keeps running.

## Looping Through All Features

A GeoJSON layer on a map is just a list of features. Looping over that list is how you inspect, filter, and transform data — **maps are just loops**.

In [ ]:
# Print the name and type of every feature
for feature in geojson["features"]:
    name      = feature["properties"]["name"]
    feat_type = feature["properties"]["type"]
    geom_type = feature["geometry"]["type"]
    print(f"{geom_type:12}  [{feat_type}]  {name}")

## Exploring Properties

Once you can loop, you can answer questions about the data. Here are some common patterns.

In [ ]:
# Count total features
print("Total features:", len(geojson["features"]))

In [ ]:
# Find all unique values of a property
types = {f["properties"]["type"] for f in geojson["features"]}
print("Feature types:", types)

In [ ]:
# Find features matching a condition
parks = [
    f["properties"]["name"]
    for f in geojson["features"]
    if f["properties"].get("type") == "park"
]
print("Parks:", parks)

In [ ]:
# Count features per type
from collections import Counter

counts = Counter(f["properties"]["type"] for f in geojson["features"])
for feat_type, count in counts.items():
    print(f"  {feat_type:12} {count}")

## Properties and Map Appearance

Properties are not just labels — they are the data you use to **drive visual decisions**.

For example, these are the standard GeoJSON styling hints (used by tools like geojson.io):

| Property key | Controls |
|---|---|
| `fill` | polygon fill color (hex) |
| `fill-opacity` | polygon fill opacity (0–1) |
| `stroke` | line/polygon border color (hex) |
| `stroke-width` | line/polygon border thickness (px) |
| `stroke-opacity` | line/polygon border opacity (0–1) |
| `marker-color` | point marker color (hex) |
| `marker-size` | point marker size (`"small"`, `"medium"`, `"large"`) |
| `marker-symbol` | point icon name (Maki icon set) |

ipyleaflet does not read these keys automatically — but you can read them yourself and pass the values into ipyleaflet's `style` argument. The next lesson shows exactly how to do that.

In [ ]:
# Add styling hints to the GeoJSON and save the result
COLOR_BY_TYPE = {
    "government": "#e74c3c",
    "education":  "#3498db",
    "water":      "#1abc9c",
    "park":       "#2ecc71",
    "route":      "#e67e22",
}

for feature in geojson["features"]:
    feat_type = feature["properties"].get("type", "unknown")
    color = COLOR_BY_TYPE.get(feat_type, "#95a5a6")
    feature["properties"]["marker-color"] = color
    feature["properties"]["stroke"]       = color
    feature["properties"]["fill"]         = color
    feature["properties"]["fill-opacity"] = 0.4

# Verify one feature
geojson["features"][0]["properties"]

The color hints are now stored in each feature's properties. In the next lesson we will write a **style function** that reads these values and tells ipyleaflet how to draw each feature.

## Exercise A

Build a dict that maps each feature `type` to a list of feature names with that type.

Expected shape: `{"park": ["Lucy Park", ...], "water": [...], ...}`

In [ ]:
from collections import defaultdict

# Build a dict mapping each feature type to a list of names
# e.g. {"park": ["Lucy Park", ...], "water": [...], ...}
# Your code here

## Exercise B

Add a new property `"label"` to every feature in `geojson["features"]`, formatted as `"{type}: {name}"` (e.g. `"park: Lucy Park"`). Print the first three labels to confirm.

In [ ]:
# Add a "label" property to every feature formatted as "{type}: {name}"
# Print the first 3 labels to confirm
# Your code here

---

## Check Your Understanding

Using `geojson["features"]`:

1. Print the `name` and geometry type of every feature whose `type` property is **not** `"park"`.
2. How many features have a `"route"` type?

```python
# your answer here
```

## Next

In [01 — Style Functions](./01-Style_Functions.ipynb), we use the properties we just read to drive visual appearance — giving each feature a color based on what it is.